# 第 15 章习题与解答

## Exercise 15.1

**题目**:解释温度 T 在蒸馏中的作用。T=1 和 T=10 的 softmax 分布有什么区别?

<details><summary><b>参考答案</b></summary>

温度 T 控制 softmax 分布的「软度」:

- **T=1**(标准 softmax):分布尖锐,top-1 token 可能占 95% 概率,其余 token 几乎为 0。student 只学到「选最优 token」,丢失了 teacher 对其他 token 的相对判断。
- **T=10**:分布平坦,top-1 降到 ~15%,其他 token 的概率显著提升。student 能看到 teacher 认为 token B 比	token C 更好(即使两者都不是最优)—— 这就是**暗知识**。

**为什么 T² 缩放?** softmax 输入除以 T 后,梯度幅度也缩小 T² 倍(链式法则)。乘以 T² 补偿回来,确保蒸馏 loss 和 CE loss 的梯度量级匹配。

</details>

## Exercise 15.2

**题目**:MoE 的 aux_loss 如何纠正 expert 负载不均衡?如果去掉 aux_loss 会怎样?

<details><summary><b>参考答案</b></summary>

**aux_loss = load × scores.mean** 的工作原理:

- `load_i` = 专家 i 实际处理的 token 比例
- `scores.mean_i` = 路由器对专家 i 的平均概率

当负载均匀(load_i ≈ 1/N for all i),aux_loss 最小。当所有 token 走同一个专家(load_k=1, 其余=0),aux_loss 最大。

**aux_loss 鼓励**:路由器的概率分布(scores.mean)与实际负载(load)一致 —— 如果一个专家实际负载很高但路由概率不高(或反之),aux_loss 增大,梯度推动两者对齐。

**如果去掉 aux_loss**:

发生**路由崩塌**:训练初期某个专家因为随机初始化略好,路由器开始偏向它 → 它得到更多训练 → 变得更好 → 路由器更偏向它 → 正反馈循环 → 最终所有 token 走同一个专家,其余专家变成死参数 → 等价于 dense 模型,白浪费了 3/4 的参数。

</details>

## Exercise 15.3

**题目**:如果要把 minimind 从 dense 蒸馏成 dense(而非 MoE→dense),teacher 和 student 应该怎么设置?还有意义吗?

<details><summary><b>参考答案</b></summary>

**Dense→Dense 蒸馏的设置**:

- Teacher:dense 模型,但更大(比如 hidden_size=1024, layers=12,即 ~150M 参数)
- Student:dense 模型,更小(hidden_size=768, layers=8,即 64M)
- 两者都用 dense 架构,只是尺寸不同

**有意义吗?** 有,但场景不同:

- **MoE→Dense**(minimind 默认):目标是**部署优化** —— MoE 太重,压成轻量 dense
- **Large-Dense→Small-Dense**:目标是**模型压缩** —— 大模型能力强但慢,蒸馏成小模型加速推理

两者都有实际价值。Hinton 原始蒸馏论文(2015)就是 Large-Dense→Small-Dense。Dark knowledge 在两种设置下都有效。

</details>